### Hot Keys:
**open a markdown chunk** esc, b, m, entr 

**collapse files tab (and reopen)** cmd-b

**clear all outputs** cmd-shft-c, type "clear", enter

In [1]:
## ▶ RUN THIS FIRST (every session, and after every kernel restart)


#          prints path of the Python running this notebook
import sys, os
print(sys.executable)

#          lets !commands and %%bash find tools in the env (e.g. sourmash)
os.environ["PATH"] = os.path.dirname(sys.executable) + ":" + os.environ["PATH"]

/home/hehouts/.conda/envs/branchwater/bin/python


## Done once (don't re-run)
### Setup branchwater kernel in jupyter notebooks Farm OnDemand

Registered the `branchwater` conda env as a Jupyter kernel, from an SSH terminal on Farm:

```bash
module load conda
eval "$(conda shell.bash hook)"        # fixes "Run 'conda init' before 'conda activate'"
conda activate branchwater
conda install -y ipykernel
python -m ipykernel install --user --name branchwater --display-name "Python (branchwater)"
```


Added pandas to the env (from the notebook, then restarted the kernel):
```python
%conda install -y -c conda-forge pandas
```


In [2]:
# Set up:

# location of parquet file on farm
# /group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet

# Remember, you need the branchwater conda env

from pathlib import Path

#%conda install -y -c conda-forge pandas
import pandas as pd

In [3]:
# Recall the list of bioProjects of interest
#     .Prints list of bio projects by reading file.
print('bioProjects_AMD.txt', "contains:")

# read list of bioProjects into session as 'my_bioprojects'
with open('bioProjects_AMD.txt') as f:
    my_bioprojects = [line.strip() for line in f if line.strip()]
my_bioprojects

bioProjects_AMD.txt contains:


['PRJNA1125351',
 'PRJNA554371',
 'PRJNA666025',
 'PRJNA699454',
 'PRJNA723739',
 'PRJNA801253',
 'PRJNA954676',
 'PRJDB19179',
 'PRJNA646106']

In [4]:
print("there is a total of [", len(my_bioprojects), "] bioProjects")

there is a total of [ 9 ] bioProjects


In [5]:
# generates a list of AMD BioProject accessions by subsetting bioProjects_AMD.txt (one accession per line, blank lines skipped).
# load metadata
import polars as pl
df = pl.read_parquet('/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet')


print("There are ", df.shape, " metagenomes in the parquet file, with that number of attributes...?")


There are  (1160375, 67)  metagenomes in the parquet file, with that number of attributes...?


In [8]:
# how many samples across bio projects?

# read list of bioProjects
with open('bioProjects_AMD.txt') as f:
    my_bioprojects = [line.strip() for line in f if line.strip()]
#print("my bioprojects are: ",my_bioprojects)


# makes subtable of just accessions for desired bioproject list. 
# OUTCOME: 103 samples
df2 = (
    df
    .filter(pl.col('bioproject').is_in(my_bioprojects))
    .select(['acc', 'sample_name', 'bioproject'])
)


#print("my bioprojects are: ",my_bioprojects)
print("across the projects, the wort collection has: ", df2.height, "samples")

across the projects, the wort collection has:  103 samples


In [20]:
# saved in sra_accessions_AMD.txt
sra_accs = df2['acc'].to_list()

with open('sra_accessions_AMD.txt', 'w') as f:
    for acc in sra_accs:
        f.write(acc + '\n')
print(f"Got {len(sra_accs)} accessions")
print("Wrote sra_accessions_AMD.txt")
print(f"test metagenome accs: {sra_accs[:3]}")  # the first 3

Got 103 accessions
Wrote sra_accessions_AMD.txt
test metagenome accs: ['SRR13197465', 'SRR13201267', 'SRR13213328']


In [21]:
%%bash
# confirm reference is here still 

ls /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip
ls /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig

/home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip
/group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig


In [ ]:
%%bash
# TESTsearch (one) metagenome, SRR13197465
# Path to metagenome /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig

# This did work!

sourmash scripts manysearch \
  -k 21 \
  --scaled 1000 \
  --threshold=0 \
  /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip \
  /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig \
  -o output_test1.csv

In [ ]:
%%bash
# TESTsearch (one) metagenome, SRR13197465
# Path to metagenome /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig

# This did work!

sourmash scripts manysearch \
  -k 21 \
  --scaled 1000 \
  --threshold=0 \
  /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip \
  /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig \
  -o output_103_AMD.csv

In [ ]:
#from pathlib import Path

#sigs_dir = Path("/group/ctbrowngrp5/wort/wort-sra/sigs")
#accessions = Path("/home/hehouts/AMD/sra_accessions_AMD.txt").read_text().splitlines()

#present = [a for a in accessions if (sigs_dir / f"{a}.sig").exists()]
##missing = [a for a in accessions if a not in present]

#if present:
#    print(f"found {len(present)} of {len(accessions)} files")
##    for a in present:
##        print(f"  {a}")
##    if missing:
##        print("no hits for:")
##        for a in missing:
##            print(f"  {a}")
#else:
#    print("struck out :/")

In [ ]:
%%bash
# Makes needed input files for sourmash manysearch


# saves the gtdb query .txt
echo '/home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip' > GTDB_singleton_queries.txt
ls *queries.txt 

# saves .txt with file paths to AMD metagenome .sig files
# with all the file paths to the found .sig files 
# this is the "against" sigs
#!!!! assumes accession.sig files have been validly located!!!!
sed 's|^|/group/ctbrowngrp5/wort/wort-sra/sigs/|; s|$|.sig|' \
  sra_accessions_AMD.txt > AMD_metagenome_against_sigs.txt
ls *against_sigs.txt

In [ ]:
%%bash
# running a set of 3
sourmash scripts manysearch \
  -k 21 \
  --scaled 1000 \
  --threshold=0 \
  -c 8\
  GTDB_singleton_queries.txt \
  AMDsub3_metagenome_against_sigs.txt\
  -o tmp_out.csv

mv tmp_out.csv output_$(date +%Y%m%d_%H%M%S).csv

In [ ]:
#NOTE
#save .ipynb as .py in command line with
#  jupyter nbconvert --to script acquire_metadata_w_branchwater_AMD.ipynb